In [ ]:
import os
import sys
os.chdir("../..")

import pandas as pd
import duckdb
from pathlib import Path
from config import DATA_ROOT, RESEARCH_ROOT

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

import numpy as np
from scipy import stats

In [ ]:
RESEARCH_DB_PATH = DATA_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)

In [5]:
# ── 1. Bootstrap CI for VIX_high + basis_mid cell ──────────────────────────
cell = con.execute("""
    SELECT fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE split = 'train'
      AND vix_close >= 18
      AND basis >= 50 AND basis < 100
      AND fwd_ret_5d IS NOT NULL
      AND fwd_ret_20d IS NOT NULL
""").df()

np.random.seed(42)
n_boot = 10000

boot_5d  = [np.mean(np.random.choice(cell['fwd_ret_5d'],  size=len(cell), replace=True)) for _ in range(n_boot)]
boot_20d = [np.mean(np.random.choice(cell['fwd_ret_20d'], size=len(cell), replace=True)) for _ in range(n_boot)]

print("=== Bootstrap CI — VIX_high + basis_mid (train) ===")
print(f"5D  return: mean={np.mean(boot_5d):.3f}%  95% CI [{np.percentile(boot_5d,2.5):.3f}, {np.percentile(boot_5d,97.5):.3f}]")
print(f"20D return: mean={np.mean(boot_20d):.3f}%  95% CI [{np.percentile(boot_20d,2.5):.3f}, {np.percentile(boot_20d,97.5):.3f}]")
print(f"P(5D > 0):  {np.mean(np.array(boot_5d) > 0):.3f}")
print(f"P(20D > 0): {np.mean(np.array(boot_20d) > 0):.3f}")

# ── 2. Regression with VIX × Basis interaction term ────────────────────────
from sklearn.linear_model import LinearRegression

df_train = con.execute("""
    SELECT 
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d,
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND cost_of_carry IS NOT NULL
      AND fut_chng_oi_pct IS NOT NULL
""").df()

# Standardize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

features = ['vix_close', 'basis', 'cost_of_carry', 'fut_chng_oi_pct']
X = df_train[features].copy()
X['vix_x_basis'] = df_train['vix_close'] * df_train['basis']  # interaction term
X_scaled = scaler.fit_transform(X)

# OLS with statsmodels for proper p-values
import statsmodels.api as sm
X_sm = sm.add_constant(X_scaled)
model = sm.OLS(df_train['fwd_ret_5d'], X_sm).fit()

print("\n=== OLS Regression — 5D Forward Return (train) ===")
coef_names = ['const'] + features + ['vix_x_basis']
for name, coef, pval in zip(coef_names, model.params, model.pvalues):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:20s}  coef={coef:+.4f}  p={pval:.4f}")

print(f"\nR²={model.rsquared:.4f}  Adj-R²={model.rsquared_adj:.4f}")

=== Bootstrap CI — VIX_high + basis_mid (train) ===
5D  return: mean=2.561%  95% CI [0.835, 4.419]
20D return: mean=4.511%  95% CI [2.486, 6.738]
P(5D > 0):  0.999
P(20D > 0): 1.000

=== OLS Regression — 5D Forward Return (train) ===
   const                 coef=+0.1423  p=0.2310
   vix_close             coef=+0.1277  p=0.5464
✅ basis                 coef=-3.7975  p=0.0002
   cost_of_carry         coef=-0.1628  p=0.2196
   fut_chng_oi_pct       coef=-0.0264  p=0.8523
✅ vix_x_basis           coef=+3.5263  p=0.0007

R²=0.2141  Adj-R²=0.1971


In [6]:
df_train2 = con.execute("""
    SELECT 
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND cost_of_carry IS NOT NULL
      AND fut_chng_oi_pct IS NOT NULL
""").df()

# Theoretically motivated transformations
df_train2['log_vix']      = np.log(df_train2['vix_close'])
df_train2['basis_sq']     = df_train2['basis'] ** 2
df_train2['log_vix_x_basis'] = df_train2['log_vix'] * df_train2['basis']

features_v2 = ['log_vix', 'basis', 'basis_sq', 'cost_of_carry', 
                'fut_chng_oi_pct', 'log_vix_x_basis']

X2 = df_train2[features_v2].copy()
X2_scaled = StandardScaler().fit_transform(X2)
X2_sm = sm.add_constant(X2_scaled)

model2 = sm.OLS(df_train2['fwd_ret_5d'], X2_sm).fit()

print("=== OLS v2 — Log VIX + Basis² + Interaction (train) ===")
coef_names2 = ['const'] + features_v2
for name, coef, pval in zip(coef_names2, model2.params, model2.pvalues):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}")

print(f"\nR²={model2.rsquared:.4f}  Adj-R²={model2.rsquared_adj:.4f}")

# Compare adjusted R² between models
print(f"\nModel 1 Adj-R²: 0.1971  (linear + VIX×basis)")
print(f"Model 2 Adj-R²: {model2.rsquared_adj:.4f}  (log VIX + basis² + interaction)")

=== OLS v2 — Log VIX + Basis² + Interaction (train) ===
   const                      coef=+0.1423  p=0.2247
   log_vix                    coef=+0.0555  p=0.7849
✅ basis                      coef=-13.4107  p=0.0000
✅ basis_sq                   coef=+1.0937  p=0.0015
   cost_of_carry              coef=+0.0358  p=0.8050
   fut_chng_oi_pct            coef=+0.1492  p=0.3223
✅ log_vix_x_basis            coef=+11.9581  p=0.0000

R²=0.2384  Adj-R²=0.2185

Model 1 Adj-R²: 0.1971  (linear + VIX×basis)
Model 2 Adj-R²: 0.2185  (log VIX + basis² + interaction)


In [29]:
PAPER_BLUE   = '#1f6f8b'
PAPER_ORANGE = '#d98c0f'
PAPER_RED    = '#b3293f'
PAPER_TEAL   = '#2a7f62'
PAPER_GOLD   = '#b8860b'
PAPER_GRAY   = '#888888'

GRID_COLOR   = '#e0e0e0'
AXIS_COLOR   = '#444444'
TEXT_COLOR   = '#1a1a1a'

In [62]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Publication style — reusable across all plots in the paper ──
PAPER_STYLE = {
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#444444',
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linewidth': 0.6,
    'grid.alpha': 0.7,
    'axes.labelcolor': '#1a1a1a',
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10.5,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'font.family': 'serif',
    'font.serif': ['Georgia', 'Times New Roman', 'DejaVu Serif'],
    'legend.frameon': True,
    'legend.facecolor': 'white',
    'legend.edgecolor': '#cccccc',
    'legend.fontsize': 9.5,
    'savefig.facecolor': 'white',
    'savefig.dpi': 300,
}
plt.rcParams.update(PAPER_STYLE)

# Muted, colorblind-conscious academic palette (replaces dark-theme neon accents)
PAPER_GOLD   = '#b8860b'
PAPER_GRAY   = '#888888'

# ── Data prep (unchanged) ──
df_plot = con.execute("""
    SELECT basis, fwd_ret_5d, ln(vix_close) as log_vix
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND basis IS NOT NULL
      AND vix_close IS NOT NULL
""").df()

p5, p95 = df_plot['basis'].quantile([0.05, 0.95])
df_plot['basis_w'] = df_plot['basis'].clip(p5, p95)

log_vix_median = df_plot['log_vix'].median()
basis_range = np.linspace(p5, p95, 200)

b1 = model2.params.iloc[2]
b2 = model2.params.iloc[3]
b3 = model2.params.iloc[4]

basis_mean = df_plot['basis_w'].mean()
basis_std  = df_plot['basis_w'].std()
basis_sq_mean = (df_plot['basis_w']**2).mean()
basis_sq_std  = (df_plot['basis_w']**2).std()
interaction_raw = df_plot['log_vix'] * df_plot['basis_w']
inter_mean = interaction_raw.mean()
inter_std  = interaction_raw.std()

basis_scaled    = (basis_range - basis_mean) / basis_std
basis_sq_scaled = (basis_range**2 - basis_sq_mean) / basis_sq_std
interaction_scaled = ((log_vix_median * basis_range) - inter_mean) / inter_std

predicted = (model2.params.iloc[0]
             + b1 * basis_scaled
             + b2 * basis_sq_scaled
             + b3 * interaction_scaled)

adj_r2 = model2.rsquared_adj
n_obs = len(df_plot)

# ── Figure ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Non-Linear Basis Effect on 5-Day Forward Returns',
             fontsize=13.5, fontweight='bold', color='#1a1a1a', y=1.02)

# Panel 1: quadratic fit
ax1.scatter(df_plot['basis'], df_plot['fwd_ret_5d'],
            alpha=0.35, color=PAPER_BLUE, s=14, edgecolors='none',
            label=f'Observed (n={n_obs})', zorder=2)
ax1.plot(basis_range, predicted, color=PAPER_ORANGE, linewidth=2.2,
         label='Conditional fit\n(at median VIX)', zorder=3)
ax1.axhline(0, color='#999999', linewidth=0.8, zorder=1)
ax1.axvline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=1)
ax1.set_xlabel('Basis (Futures − Spot)')
ax1.set_ylabel('5-Day Forward Return (%)')
ax1.set_title('(a) Quadratic Basis Effect', loc='left', fontsize=11)
ax1.legend(loc='upper right', fontsize=9)
ax1.text(0.02, 0.02, f'Adj. R² = {adj_r2:.3f}',
         transform=ax1.transAxes, fontsize=9, color='#555555',
         verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                    edgecolor='#cccccc', linewidth=0.6))

# Panel 2: VIX regime overlay
colors = {
    'Low VIX (<14)':   ('#1f6f8b', 'o'),   # strong teal-blue, circle
    'Mid VIX (14–18)': ('#d98c0f', '^'),   # strong amber, triangle
    'High VIX (≥18)':  ('#b3293f', 's'),   # strong red, square
}

for regime, (color, marker) in colors.items():
    if regime.startswith('Low'):
        mask = df_plot['log_vix'] < np.log(14)
    elif regime.startswith('Mid'):
        mask = (df_plot['log_vix'] >= np.log(14)) & (df_plot['log_vix'] < np.log(18))
    else:
        mask = df_plot['log_vix'] >= np.log(18)
    ax2.scatter(df_plot.loc[mask, 'basis'], df_plot.loc[mask, 'fwd_ret_5d'],
                alpha=0.75, color=color, s=32, marker=marker,
                edgecolors='white', linewidths=0.5,
                label=regime, zorder=3 if regime.startswith('High') else 2)

ax2.axhline(0, color='#999999', linewidth=0.8, zorder=1)
ax2.axvline(0, color='#999999', linewidth=0.8, linestyle='--', zorder=1)
ax2.set_xlabel('Basis (Futures − Spot)')
ax2.set_ylabel('5-Day Forward Return (%)')
ax2.set_title('(b) Basis–Return Relationship by VIX Level', loc='left', fontsize=11)
ax2.legend(loc='upper right', fontsize=9, title='VIX Regime', title_fontsize=9.5)

for ax in (ax1, ax2):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(RESEARCH_ROOT / 'plots/basis_analysis.png', dpi=300, bbox_inches='tight')
print("saved")

saved


In [8]:
# Check if the basis² significance is driven by discount outliers
print("Basis distribution in train:")
print(df_plot['basis'].describe())
print(f"\nDays with basis < 0: {(df_plot['basis'] < 0).sum()}")
print(f"Days with basis > 150: {(df_plot['basis'] > 150).sum()}")

Basis distribution in train:
count    249.000000
mean      58.037349
std       46.828927
min      -32.600000
25%       20.450000
50%       57.300000
75%       85.300000
max      225.600000
Name: basis, dtype: float64

Days with basis < 0: 29
Days with basis > 150: 14


In [9]:
from scipy.stats.mstats import winsorize

# Check 5th and 95th percentiles
p5  = df_plot['basis'].quantile(0.05)
p95 = df_plot['basis'].quantile(0.95)
print(f"Basis p5={p5:.1f}, p95={p95:.1f}")

# Rerun model with winsorized basis
df_train3 = con.execute("""
    SELECT 
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d
    FROM daily_features
    WHERE split = 'train'
      AND fwd_ret_5d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND cost_of_carry IS NOT NULL
      AND fut_chng_oi_pct IS NOT NULL
""").df()

df_train3['log_vix']       = np.log(df_train3['vix_close'])
df_train3['basis_w']       = df_train3['basis'].clip(lower=p5, upper=p95)
df_train3['basis_sq_w']    = df_train3['basis_w'] ** 2
df_train3['log_vix_x_basis_w'] = df_train3['log_vix'] * df_train3['basis_w']

features_v3 = ['log_vix', 'basis_w', 'basis_sq_w', 
                'cost_of_carry', 'fut_chng_oi_pct', 'log_vix_x_basis_w']

X3 = df_train3[features_v3].copy()
scaler3 = StandardScaler()
X3_scaled = scaler3.fit_transform(X3)
X3_sm = sm.add_constant(X3_scaled)

model3 = sm.OLS(df_train3['fwd_ret_5d'], X3_sm).fit()

print("\n=== OLS v3 — Winsorized Basis (train) ===")
coef_names3 = ['const'] + features_v3
for name, coef, pval in zip(coef_names3, model3.params, model3.pvalues):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}")

print(f"\nR²={model3.rsquared:.4f}  Adj-R²={model3.rsquared_adj:.4f}")
print(f"vs Model 1 Adj-R²: 0.1971")
print(f"vs Model 2 Adj-R²: 0.2185")

Basis p5=-6.3, p95=153.9

=== OLS v3 — Winsorized Basis (train) ===
   const                      coef=+0.1423  p=0.2233
   log_vix                    coef=-0.0207  p=0.9225
✅ basis_w                    coef=-13.7086  p=0.0000
✅ basis_sq_w                 coef=+1.2277  p=0.0018
   cost_of_carry              coef=+0.0330  p=0.8165
   fut_chng_oi_pct            coef=+0.1661  p=0.2657
✅ log_vix_x_basis_w          coef=+12.0885  p=0.0000

R²=0.2430  Adj-R²=0.2233
vs Model 1 Adj-R²: 0.1971
vs Model 2 Adj-R²: 0.2185


In [10]:
features_final = ['basis_w', 'basis_sq_w', 'log_vix_x_basis_w']

X_final = df_train3[features_final].copy()
scaler_final = StandardScaler()
X_final_scaled = scaler_final.fit_transform(X_final)
X_final_sm = sm.add_constant(X_final_scaled)

model_final = sm.OLS(df_train3['fwd_ret_5d'], X_final_sm).fit()

print("=== Final Parsimonious Model (train) ===")
for name, coef, pval, ci_low, ci_high in zip(
    ['const'] + features_final,
    model_final.params,
    model_final.pvalues,
    model_final.conf_int()[0],
    model_final.conf_int()[1]
):
    sig = "✅" if pval < 0.05 else "  "
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}  95%CI=[{ci_low:+.4f}, {ci_high:+.4f}]")

print(f"\nAdj-R²={model_final.rsquared_adj:.4f}")
print(f"AIC={model_final.aic:.2f}")
print(f"BIC={model_final.bic:.2f}")

# Also run Durbin-Watson for autocorrelation (important for time series)
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(model_final.resid)
print(f"Durbin-Watson={dw:.4f}  (2.0=no autocorrelation, <2=positive autocorr)")

=== Final Parsimonious Model (train) ===
   const                      coef=+0.1423  p=0.2217  95%CI=[-0.0865, +0.3710]
✅ basis_w                    coef=-13.2832  p=0.0000  95%CI=[-16.5607, -10.0056]
✅ basis_sq_w                 coef=+1.1078  p=0.0015  95%CI=[+0.4266, +1.7889]
✅ log_vix_x_basis_w          coef=+11.8607  p=0.0000  95%CI=[+8.7760, +14.9454]

Adj-R²=0.2288
AIC=951.78
BIC=965.65
Durbin-Watson=0.5545  (2.0=no autocorrelation, <2=positive autocorr)


In [11]:
# HAC (Newey-West) standard errors — correct for autocorrelation and heteroskedasticity
# Standard in financial econometrics, expected by reviewers
model_hac = sm.OLS(df_train3['fwd_ret_5d'], X_final_sm).fit(
    cov_type='HAC', 
    cov_kwds={'maxlags': 5}  # 5 lags covers one trading week
)

print("=== Final Model with HAC Standard Errors (Newey-West, 5 lags) ===")
for name, coef, pval, ci_low, ci_high in zip(
    ['const'] + features_final,
    model_hac.params,
    model_hac.pvalues,
    model_hac.conf_int()[0],
    model_hac.conf_int()[1]
):
    sig = "✅" if pval < 0.05 else "❌"
    print(f"{sig} {name:25s}  coef={coef:+.4f}  p={pval:.4f}  95%CI=[{ci_low:+.4f}, {ci_high:+.4f}]")

print(f"\nAdj-R²={model_hac.rsquared_adj:.4f}")

# Also check how bad the autocorrelation is at different lags
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(model_final.resid, lags=[1, 5, 10], return_df=True)
print("\n=== Ljung-Box Test (residual autocorrelation) ===")
print(lb_test)
print("(p < 0.05 means significant autocorrelation at that lag)")

=== Final Model with HAC Standard Errors (Newey-West, 5 lags) ===
❌ const                      coef=+0.1423  p=0.4941  95%CI=[-0.2655, +0.5500]
✅ basis_w                    coef=-13.2832  p=0.0000  95%CI=[-19.6600, -6.9063]
✅ basis_sq_w                 coef=+1.1078  p=0.0238  95%CI=[+0.1470, +2.0686]
✅ log_vix_x_basis_w          coef=+11.8607  p=0.0001  95%CI=[+5.8563, +17.8651]

Adj-R²=0.2288

=== Ljung-Box Test (residual autocorrelation) ===
       lb_stat     lb_pvalue
1   125.008647  5.067338e-29
5   222.125961  5.206316e-46
10  232.763010  2.262212e-44
(p < 0.05 means significant autocorrelation at that lag)


In [78]:
# Plot residual diagnostics — paper style
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Assumes PAPER_STYLE rcParams + PAPER_BLUE/PAPER_ORANGE/PAPER_RED/PAPER_TEAL/PAPER_GOLD
# already applied earlier in the notebook (see basis_analysis.png setup).

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Residual Diagnostics', fontsize=13.5, fontweight='bold',
             color='#1a1a1a', y=1.02)

resid = model_final.resid
df_train3_sorted = df_train3.copy().reset_index(drop=True)
df_train3_sorted['resid'] = resid.values

# (a) Residual ACF
plot_acf(resid, lags=30, ax=axes[0, 0], color=PAPER_BLUE,
         zero=False, title="")
axes[0, 0].axhline(0, color='#999999', linewidth=0.8)
axes[0, 0].set_title('(a) Residual Autocorrelation Function', loc='left', fontsize=11)
axes[0, 0].set_xlabel('Lag')
axes[0, 0].set_ylabel('ACF')

# (b) Normal Q-Q Plot
stats.probplot(resid, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('')
axes[0, 1].get_lines()[0].set_markerfacecolor(PAPER_BLUE)
axes[0, 1].get_lines()[0].set_markeredgecolor(PAPER_BLUE)
axes[0, 1].get_lines()[0].set_alpha(0.55)
axes[0, 1].get_lines()[0].set_markersize(4)
axes[0, 1].get_lines()[1].set_color(PAPER_ORANGE)
axes[0, 1].get_lines()[1].set_linewidth(2.2)
axes[0, 1].set_title('(b) Normal Q-Q Plot', loc='left', fontsize=11)
axes[0, 1].set_xlabel('Theoretical Quantiles')
axes[0, 1].set_ylabel('Sample Quantiles')

# (c) Residuals vs VIX + LOWESS trend
axes[1, 0].scatter(df_train3_sorted['vix_close'], df_train3_sorted['resid'],
                    color=PAPER_BLUE, alpha=0.45, s=18, edgecolors='none', zorder=2)
smooth = lowess(df_train3_sorted['resid'], df_train3_sorted['vix_close'], frac=0.3)
axes[1, 0].plot(smooth[:, 0], smooth[:, 1], color=PAPER_ORANGE,
                 linewidth=2.2, label='LOWESS trend', zorder=3)
axes[1, 0].axhline(0, color='#999999', linewidth=0.8, zorder=1)
axes[1, 0].set_title('(c) Residuals vs VIX', loc='left', fontsize=11)
axes[1, 0].set_xlabel('VIX')
axes[1, 0].set_ylabel('Residual')
axes[1, 0].legend(loc='upper right', fontsize=9, frameon=True,
                   facecolor='white', edgecolor='#cccccc')

# (d) Rolling 20-Day Residual Mean
rolling_resid = pd.Series(resid.values).rolling(20).mean()
x_idx = np.arange(len(rolling_resid))
axes[1, 1].plot(x_idx, rolling_resid, color=PAPER_GOLD, linewidth=1.8, zorder=3)
axes[1, 1].fill_between(x_idx, rolling_resid, 0,
                         where=rolling_resid > 0, alpha=0.3, color=PAPER_TEAL, zorder=2)
axes[1, 1].fill_between(x_idx, rolling_resid, 0,
                         where=rolling_resid < 0, alpha=0.3, color=PAPER_RED, zorder=2)
axes[1, 1].axhline(0, color='#999999', linewidth=0.8, zorder=1)
axes[1, 1].set_title('(d) Rolling 20-Day Residual Mean', loc='left', fontsize=11)
axes[1, 1].set_xlabel('Trading Day')
axes[1, 1].set_ylabel('Residual (20D rolling mean)')

for ax in axes.flatten():
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(RESEARCH_ROOT / 'plots/residual_diagnostics.png',
            dpi=300, bbox_inches='tight', facecolor='white')
print("saved")

saved


In [13]:
import subprocess
subprocess.run(['pip', 'install', 'hmmlearn', '--break-system-packages', '-q'])

from hmmlearn import hmm

# Pull training data for HMM
df_hmm = con.execute("""
    SELECT
        trade_date,
        vix_close,
        basis,
        cost_of_carry,
        fut_chng_oi_pct,
        fwd_ret_5d
    FROM daily_features
    WHERE split = 'train'
    AND fwd_ret_5d IS NOT NULL
    AND vix_close IS NOT NULL
    AND basis IS NOT NULL
    AND cost_of_carry IS NOT NULL
    AND fut_chng_oi_pct IS NOT NULL
    ORDER BY trade_date
""").df()

print(f"HMM training observations: {len(df_hmm)}")
print(df_hmm.describe().round(3))

HMM training observations: 237
                       trade_date  vix_close    basis  cost_of_carry  \
count                         237    237.000  237.000        237.000   
mean   2024-12-20 03:56:57.721519     14.861   61.075          0.066   
min           2024-06-21 00:00:00     11.762  -32.600         -0.370   
25%           2024-09-19 00:00:00     13.670   27.300          0.037   
50%           2024-12-19 00:00:00     14.395   59.300          0.064   
75%           2025-03-19 00:00:00     15.650   87.650          0.085   
max           2025-06-20 00:00:00     22.792  225.600          0.545   
std                           NaN      1.870   45.906          0.082   

       fut_chng_oi_pct  fwd_ret_5d  
count          237.000     237.000  
mean            -3.171       0.142  
min            -39.327      -5.773  
25%             -3.430      -1.138  
50%             -1.418       0.190  
75%              1.049       1.504  
max              9.916       7.707  
std              8.041  

In [14]:
print("df_hmm shape:", df_hmm.shape)

# Check nulls in source
check = con.execute("""
    SELECT 
        COUNT(*) as total,
        COUNT(vix_close) as has_vix,
        COUNT(basis) as has_basis,
        COUNT(fut_chng_oi_pct) as has_oi,
    FROM daily_features
    WHERE split = 'train'
""").df()
print(check)

# Find the missing dates
all_dates = con.execute("""
    SELECT trade_date FROM daily_features 
    WHERE split = 'train' ORDER BY trade_date
""").df()

print(f"\nTotal train rows: {len(all_dates)}")
print(f"After null filter: {len(df_hmm)}")
print(f"Missing: {len(all_dates) - len(df_hmm)}")

# Which dates are missing
missing = all_dates[~all_dates['trade_date'].isin(df_hmm['trade_date'])]
print("\nMissing dates:")
print(missing)

df_hmm shape: (237, 6)
   total  has_vix  has_basis  has_oi
0    249      249        249     249

Total train rows: 249
After null filter: 237
Missing: 12

Missing dates:
    trade_date
4   2024-06-27
23  2024-07-25
47  2024-08-29
67  2024-09-26
91  2024-10-31
109 2024-11-28
128 2024-12-26
153 2025-01-30
173 2025-02-27
192 2025-03-27
208 2025-04-24
232 2025-05-29


In [15]:
df_hmm_full = con.execute("""
    SELECT trade_date, vix_close, basis, fut_chng_oi_pct
    FROM daily_features
    WHERE split = 'train'
    ORDER BY trade_date
""").df()

# Forward fill expiry nulls only
df_hmm_full['fut_chng_oi_pct'] = df_hmm_full['fut_chng_oi_pct'].ffill()

# Verify
print("Nulls remaining:", df_hmm_full['fut_chng_oi_pct'].isna().sum())
print("Shape:", df_hmm_full.shape)  # should be (249, 4)

df_hmm_full['log_vix'] = np.log(df_hmm_full['vix_close'])
df_hmm_full['basis_w'] = df_hmm_full['basis'].clip(
    df_hmm_full['basis'].quantile(0.05),
    df_hmm_full['basis'].quantile(0.95)
)

features_hmm = ['log_vix', 'basis_w', 'fut_chng_oi_pct']
scaler_hmm = StandardScaler()
X_scaled = scaler_hmm.fit_transform(df_hmm_full[features_hmm])

print("X_scaled shape:", X_scaled.shape)  # must be (249, 3)
print("means:", X_scaled.mean(axis=0).round(10))
print("stds: ", X_scaled.std(axis=0).round(4))

Nulls remaining: 0
Shape: (249, 4)
X_scaled shape: (249, 3)
means: [-0. -0. -0.]
stds:  [1. 1. 1.]


In [16]:
print("=== Canonical HMM State Selection ===")

all_results = []

for cov_type in ['diag', 'full']:
    print(f"\n-- covariance_type='{cov_type}' --")
    
    for n in range(2, 7):
        best_model = None
        best_logL = -np.inf
        
        for seed in range(50):
            try:
                model = hmm.GaussianHMM(
                    n_components=n,
                    covariance_type=cov_type,
                    n_iter=500,
                    random_state=seed,
                    tol=1e-5,
                    init_params='stmc'
                )
                model.fit(X_scaled)
                logL = model.score(X_scaled)

                if np.any(model.transmat_.max(axis=1) > 0.99):
                    continue
                _, states = model.decode(X_scaled)
                if np.any(np.bincount(states, minlength=n) < 0.10 * len(X_scaled)):
                    continue
                if logL > best_logL:
                    best_logL = logL
                    best_model = model
            except:
                continue

        if best_model is None:
            print(f"  n={n}  no valid solution found")
            continue

        d = len(features_hmm)
        cov_params = n * d * (d+1) // 2 if cov_type == 'full' else n * d
        n_params = n * (n-1) + n * d + cov_params
        bic = -2 * best_logL + n_params * np.log(len(X_scaled))

        _, states = best_model.decode(X_scaled)
        occupancy = (np.bincount(states, minlength=n) / len(X_scaled) * 100).round(1)

        print(f"  n={n}  logL={best_logL:.2f}  n_params={n_params:3d}"
              f"  BIC={bic:.2f}  occupancy={occupancy}")

        all_results.append({
            'cov': cov_type, 'n': n, 'bic': bic,
            'logL': best_logL, 'model': best_model,
            'states': states
        })

valid = [r for r in all_results if r['model'] is not None]
best = min(valid, key=lambda x: x['bic'])
print(f"\n=== FINAL: cov={best['cov']}, n={best['n']}, BIC={best['bic']:.2f} ===")

=== Canonical HMM State Selection ===

-- covariance_type='diag' --


Model is not converging.  Current: -817.614885904242 is not greater than -817.6148852801404. Delta is -6.241016308194958e-07
Model is not converging.  Current: -817.6148857610378 is not greater than -817.6148854569312. Delta is -3.041066065634368e-07
Model is not converging.  Current: -817.6148857376531 is not greater than -817.614885507477. Delta is -2.3017616967990762e-07
Model is not converging.  Current: -817.6148864223574 is not greater than -817.6148849359075. Delta is -1.4864498325550812e-06
Model is not converging.  Current: -733.6666059337905 is not greater than -733.6666041790924. Delta is -1.75469813257223e-06
Model is not converging.  Current: -733.6666060324634 is not greater than -733.6666041100559. Delta is -1.9224074776502675e-06


  n=2  logL=-817.61  n_params= 14  BIC=1712.47  occupancy=[24.5 75.5]
  n=3  logL=-728.31  n_params= 24  BIC=1589.04  occupancy=[24.9 39.  36.1]


Model is not converging.  Current: -721.2921709897757 is not greater than -721.2919899436519. Delta is -0.00018104612388469832
Model is not converging.  Current: -670.7149901812055 is not greater than -670.7149825492966. Delta is -7.631908943039889e-06
Model is not converging.  Current: -719.6272883368958 is not greater than -719.6272674675201. Delta is -2.08693757031142e-05
Model is not converging.  Current: -662.5121771774134 is not greater than -662.5121666409722. Delta is -1.0536441209296754e-05
Model is not converging.  Current: -662.5122673863258 is not greater than -662.5122588979698. Delta is -8.488356002089859e-06
Model is not converging.  Current: -662.5121725776962 is not greater than -662.5121668924658. Delta is -5.685230462404434e-06
Model is not converging.  Current: -671.1133637411044 is not greater than -671.1133603685264. Delta is -3.3725780212989775e-06
Model is not converging.  Current: -670.7144891904779 is not greater than -670.7144883661124. Delta is -8.2436554293

  n=4  logL=-662.51  n_params= 36  BIC=1523.65  occupancy=[21.3 23.3 16.5 39. ]


...

  n=4  logL=-647.81  n_params= 48  BIC=1560.46  occupancy=[13.3 16.9 36.5 33.3]


Model is not converging.  Current: -616.548022300168 is not greater than -616.5480102198261. Delta is -1.208034188948659e-05
Model is not converging.  Current: -664.357071006401 is not greater than -664.3570333961045. Delta is -3.761029654469894e-05
Model is not converging.  Current: -602.2162953080802 is not greater than -602.2162931992874. Delta is -2.1087928416818613e-06
Model is not converging.  Current: -777.4578650208699 is not greater than -775.1210260319419. Delta is -2.3368389889279797


  n=5  logL=-601.68  n_params= 65  BIC=1562.00  occupancy=[35.3 12.9 15.3 13.7 22.9]


Model is not converging.  Current: -580.3234310551913 is not greater than -580.3234262631863. Delta is -4.792005029230495e-06
Model is not converging.  Current: -584.0622378824913 is not greater than -584.0618922306268. Delta is -0.00034565186456347874
Model is not converging.  Current: -587.8254070019732 is not greater than -587.8254011348291. Delta is -5.867144068361085e-06
Model is not converging.  Current: -627.2400803362244 is not greater than -627.2400221582071. Delta is -5.817801729790517e-05
Model is not converging.  Current: -623.5684091438974 is not greater than -623.5683945955589. Delta is -1.4548338526765292e-05
Model is not converging.  Current: -598.8550031149293 is not greater than -598.854955119085. Delta is -4.799584428383241e-05


  n=6  logL=-573.81  n_params= 84  BIC=1611.09  occupancy=[20.5 11.6 14.9 25.7 14.5 12.9]

=== FINAL: cov=diag, n=5, BIC=1499.73 ===


In [17]:
import pickle

# Sanity check before saving — confirm this really is the locked canonical model
print(f"cov={best['cov']}, n={best['n']}, BIC={best['bic']:.2f}")
# Should print: cov=diag, n=5, BIC=1499.73 (per your locked notes)

(RESEARCH_ROOT / 'models').mkdir(exist_ok=True)

with open(RESEARCH_ROOT / 'models/hmm_canonical.pkl', 'wb') as f:
    pickle.dump({
        'hmm_model': best['model'],
        'scaler': scaler_hmm,
        'basis_p5': p5,
        'basis_p95': p95,
        'features_hmm': features_hmm,   # ['log_vix', 'basis_w', 'fut_chng_oi_pct'] — order matters
    }, f)

print("Saved canonical HMM model + scaler + winsorization bounds")

cov=diag, n=5, BIC=1499.73
Saved canonical HMM model + scaler + winsorization bounds


In [23]:
best5d = next(r for r in valid if r['cov'] == 'diag' and r['n'] == 5)
m5 = best5d['model']
s5 = best5d['states']

print("Transition matrix:")
print(np.round(m5.transmat_, 3))
print("\nMeans (log_vix, basis_w, fut_chng_oi_pct):")
for i, mean in enumerate(m5.means_):
    print(f"  S{i}: {np.round(mean, 3)}  n={( s5==i).sum()}")
print("\nSelf-transition probs:")
for i in range(5):
    print(f"  S{i}: {m5.transmat_[i,i]:.3f}")

Transition matrix:
[[0.718 0.    0.025 0.256 0.   ]
 [0.311 0.689 0.    0.    0.   ]
 [0.    0.059 0.878 0.    0.063]
 [0.    0.    0.024 0.851 0.125]
 [0.    0.167 0.02  0.    0.813]]

Means (log_vix, basis_w, fut_chng_oi_pct):
  S0: [-0.176  1.484  0.575]  n=43
  S1: [-0.082 -1.12  -2.033]  n=39
  S2: [ 1.761 -0.017  0.271]  n=33
  S3: [-0.407  0.373  0.424]  n=72
  S4: [-0.293 -0.764  0.225]  n=62

Self-transition probs:
  S0: 0.718
  S1: 0.689
  S2: 0.878
  S3: 0.851
  S4: 0.813


In [56]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Attach states to df_hmm_full ─────────────────────────────────────────────
df_hmm_full['state'] = best5d['states']

# ── Save to research.db ──────────────────────────────────────────────────────
con.execute("""
    ALTER TABLE daily_features
    ADD COLUMN IF NOT EXISTS hmm_state INTEGER;
""")

for _, row in df_hmm_full.iterrows():
    con.execute("""
        UPDATE daily_features
        SET hmm_state = ?
        WHERE trade_date = ?
          AND split = 'train'
    """, [int(row['state']), row['trade_date']])

print("States saved to daily_features")

results = con.execute("""
    SELECT
        hmm_state,
        COUNT(*) AS days,
        ROUND(AVG(fwd_ret_5d), 3) AS avg_5d,
        ROUND(
            AVG(
                CASE WHEN fwd_ret_5d > 0
                     THEN 1.0
                     ELSE 0.0
                END
            ) * 100,
            1
        ) AS pct_up
    FROM daily_features
    WHERE split = 'train'
      AND hmm_state IS NOT NULL
    GROUP BY hmm_state
    ORDER BY hmm_state
""").fetchall()

for row in results:
    print(row)

# ── Paper regime palette ─────────────────────────────────────────────────────
PAPER_PURPLE = '#7a6aa6'  # muted lavender — extension to the core 5-color
                          # palette, used only for this 5-state HMM figure

STATE_COLORS = {
    0: PAPER_BLUE,
    1: PAPER_RED,
    2: PAPER_ORANGE,
    3: PAPER_TEAL,    # teal, not green
    4: PAPER_PURPLE,
}

STATE_LABELS = {
    0: 'S0: High Premium',
    1: 'S1: Hard Unwind',
    2: 'S2: Stress / VIX Spike',
    3: 'S3: Drift / Accumulation',
    4: 'S4: Rolldown / Decay',
}

# Neutral foreground line color — used for ALL data series drawn over
# regime-shaded backgrounds, so color is reserved exclusively for regime
# identity and never collides with it.
REGIME_LINE_COLOR = '#444444'

# ── Figure Layout ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    4,
    1,
    figsize=(16, 10),
    gridspec_kw={'height_ratios': [1.2, 2, 1.3, 1.3]}
)

fig.suptitle(
    'Hidden Markov Model Regimes and Market Dynamics',
    fontsize=15,
    fontweight='bold',
    color='#1a1a1a',
    y=0.995
)

dates = df_hmm_full['trade_date'].values
states = df_hmm_full['state'].values

# ── Common Formatting ────────────────────────────────────────────────────────
for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ============================================================================
# (a) Hidden State Timeline
# ============================================================================
ax = axes[0]

ax.step(
    dates,
    states,
    where='post',
    color='#999999',
    linewidth=1.2,
    zorder=1
)

ax.scatter(
    dates,
    states,
    c=[STATE_COLORS[s] for s in states],
    s=14,
    zorder=2
)

ax.set_yticks([0, 1, 2, 3, 4])
ax.set_yticklabels(['S0', 'S1', 'S2', 'S3', 'S4'])

ax.set_ylabel('State')
ax.set_title(
    '(a) Inferred Hidden Market Regimes',
    loc='left',
    fontsize=11
)

patches = [
    mpatches.Patch(
        color=STATE_COLORS[s],
        label=STATE_LABELS[s]
    )
    for s in range(5)
]

ax.legend(
    handles=patches,
    bbox_to_anchor=(1.01, 1.0),
    loc='upper left',
    fontsize=8,
    frameon=True,
    facecolor='white',
    edgecolor='#cccccc',
    borderaxespad=0
)

# ============================================================================
# (b) VIX with Regime Overlay
# ============================================================================
ax = axes[1]

for i in range(len(dates) - 1):
    ax.axvspan(
        dates[i],
        dates[i + 1],
        color=STATE_COLORS[states[i]],
        alpha=0.35,
        linewidth=0
    )

ax.plot(
    dates,
    df_hmm_full['vix_close'],
    color=REGIME_LINE_COLOR,
    linewidth=1.2,
    zorder=3
)

ax.set_ylabel('VIX')
ax.set_title(
    '(b) VIX Index Across Hidden Regimes',
    loc='left',
    fontsize=11
)

# ============================================================================
# (c) Futures Basis
# ============================================================================
ax = axes[2]

for i in range(len(dates) - 1):
    ax.axvspan(
        dates[i],
        dates[i + 1],
        color=STATE_COLORS[states[i]],
        alpha=0.35,
        linewidth=0
    )

ax.plot(
    dates,
    df_hmm_full['basis'],
    color=REGIME_LINE_COLOR,
    linewidth=1.2,
    zorder=3
)

ax.axhline(
    0,
    color='#999999',
    linewidth=0.8
)

ax.set_ylabel('Basis')
ax.set_title(
    '(c) Futures Basis by Regime',
    loc='left',
    fontsize=11
)

# ============================================================================
# (d) Open Interest Change
# ============================================================================
ax = axes[3]

for i in range(len(dates) - 1):
    ax.axvspan(
        dates[i],
        dates[i + 1],
        color=STATE_COLORS[states[i]],
        alpha=0.35,
        linewidth=0
    )

ax.plot(
    dates,
    df_hmm_full['fut_chng_oi_pct'],
    color=REGIME_LINE_COLOR,
    linewidth=1.2,
    zorder=3
)

ax.axhline(
    0,
    color='#999999',
    linewidth=0.8
)

ax.set_ylabel('OI Change (%)')
ax.set_title(
    '(d) Futures Open Interest Change by Regime',
    loc='left',
    fontsize=11
)

# ── Layout ───────────────────────────────────────────────────────────────────
plt.tight_layout(rect=[0, 0, 0.92, 0.97])

plt.savefig(
    RESEARCH_ROOT / 'plots/hmm_regimes.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("Saved: hmm_regimes.png")

States saved to daily_features
(0, 43, -0.548, 44.2)
(1, 39, 0.317, 56.4)
(2, 33, 1.079, 66.7)
(3, 72, -0.48, 37.5)
(4, 62, 0.759, 74.2)
Saved: hmm_regimes.png


C:\Users\sriva\AppData\Local\Temp\ipykernel_4384\1389213174.py:256: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [55]:
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

# Pull train data with hmm_state
df = con.execute("""
    SELECT trade_date, hmm_state,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE split = 'train' AND hmm_state IS NOT NULL
    ORDER BY trade_date
""").df()

state_labels = {
    0: 'S0: HighPrem\n(n=43)',
    1: 'S1: HardUnwind\n(n=39)',
    2: 'S2: Stress\n(n=33)',
    3: 'S3: Drift\n(n=72)',
    4: 'S4: Rolldown\n(n=62)',
    'ALL': 'Full Sample\n(n=249)',
}

factors = {
    'VIX':         'vix_close',
    'PCR':         'pcr',
    'MaxPainDist': 'max_pain_dist_pct',
    'Basis':       'basis',
    'CostOfCarry': 'cost_of_carry',
    'FutOIChng':   'fut_chng_oi_pct',
    'FII_Fut':     'fii_fut_net_pct',
    'Client_Fut':  'client_fut_net_pct',
}
horizons = {'1D': 'fwd_ret_1d', '5D': 'fwd_ret_5d', '20D': 'fwd_ret_20d'}
state_names = {
    0: 'S0:HighPrem',
    1: 'S1:HardUnwind',
    2: 'S2:Stress',
    3: 'S3:Drift',
    4: 'S4:Rolldown',
}

def spearman_ic(x, y):
    mask = x.notna() & y.notna()
    if mask.sum() < 10:
        return np.nan, np.nan
    ic, p = stats.spearmanr(x[mask], y[mask])
    return ic, p

def sig_star(p):
    if np.isnan(p):
        return ''
    if p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.10:
        return '†'
    return ''

# separators now at every 3 rows for 6 groups
separator_rows = [3, 6, 9, 12, 15]

# Build IC table
results = {}
for state_key, state_label in state_labels.items():
    for hz_name, hz_col in horizons.items():
        row_key = (state_label, hz_name)
        results[row_key] = {}
        sub = df if state_key == 'ALL' else df[df['hmm_state'] == state_key]
        for fac_name, fac_col in factors.items():
            results[row_key][fac_name] = spearman_ic(sub[fac_col], sub[hz_col])

row_order = [(sl, hz) for sl in state_labels.values() for hz in horizons]
col_order  = list(factors.keys())

ic_mat = np.full((len(row_order), len(col_order)), np.nan)
p_mat  = np.full((len(row_order), len(col_order)), np.nan)
for i, rk in enumerate(row_order):
    for j, fac in enumerate(col_order):
        ic_mat[i, j], p_mat[i, j] = results[rk][fac]

# ── Paper diverging palette: built from the locked red/blue, no green ──────
PAPER_DIVERGING = LinearSegmentedColormap.from_list(
    'paper_diverging', [PAPER_RED, '#ffffff', PAPER_BLUE]
)

fig, ax = plt.subplots(figsize=(14, 12))

vmax = 0.55
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im = ax.imshow(ic_mat, cmap=PAPER_DIVERGING, norm=norm, aspect='auto')

for i in range(len(row_order)):
    for j in range(len(col_order)):
        ic_val, p_val = ic_mat[i, j], p_mat[i, j]
        if np.isnan(ic_val):
            txt, color = 'n/a', PAPER_GRAY
        else:
            txt = f'{ic_val:+.2f}{sig_star(p_val)}'
            color = 'white' if abs(ic_val) > 0.25 else '#1a1a1a'
        ax.text(j, i, txt, ha='center', va='center',
                fontsize=13, color=color, fontweight='bold')

ax.set_xticks(range(len(col_order)))
ax.set_xticklabels(col_order, color='#1a1a1a', fontsize=11, ha='center')
ax.set_yticks(range(len(row_order)))
ax.set_yticklabels([f'{sl.split(chr(10))[0]}  {hz}' for sl, hz in row_order],
                   color='#1a1a1a', fontsize=11)

# Thin white cell borders (minor-tick grid trick) — major grid turned off
# so the global PAPER_STYLE gridlines don't cut through cell centers.
ax.grid(which='major', visible=False)
ax.set_xticks(np.arange(-0.5, len(col_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=0.6)
ax.tick_params(which='minor', bottom=False, left=False)

# Group separators — heavier than the cell grid, distinct purpose
for i in separator_rows:
    ax.axhline(i - 0.5, color='#444444', linewidth=1.2)

cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
cbar.ax.yaxis.set_tick_params(color='#444444')
cbar.ax.set_ylabel('Spearman IC', color='#1a1a1a', fontsize=11)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#1a1a1a', fontsize=11)
cbar.outline.set_edgecolor('#cccccc')

ax.tick_params(colors='#444444', which='major')
for spine in ax.spines.values():
    spine.set_edgecolor('#444444')
    spine.set_linewidth(0.8)

ax.set_title('Regime-Conditioned Factor IC — HMM States vs Full Sample\n'
             '** p<0.01  * p<0.05  † p<0.10  |  Train Set Only',
             loc='center', color='#1a1a1a', fontsize=15, pad=14)

plt.tight_layout()
plt.savefig(RESEARCH_ROOT / 'plots/hmm_regime_ic_heatmap.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")

done


C:\Users\sriva\AppData\Local\Temp\ipykernel_4384\314601327.py:142: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [76]:
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

df = con.execute("""
    SELECT trade_date, hmm_state,
           vix_close, pcr, max_pain_dist_pct, basis, cost_of_carry,
           fut_chng_oi_pct, fii_fut_net_pct, client_fut_net_pct,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d, split
    FROM daily_features
    WHERE split = 'train' AND hmm_state IS NOT NULL
    ORDER BY trade_date
""").df()

# Compute full-sample and per-state ICs
records = []
for hz_name, hz_col in horizons.items():
    for fac_name, fac_col in factors.items():
        ic_full, p_full = spearman_ic(df[fac_col], df[hz_col])
        records.append({
            'factor': fac_name, 'horizon': hz_name,
            'state': 'Full', 'ic': ic_full, 'p': p_full
        })
        for state_id, state_label in state_names.items():
            sub = df[df['hmm_state'] == state_id]
            ic, p = spearman_ic(sub[fac_col], sub[hz_col])
            records.append({
                'factor': fac_name, 'horizon': hz_name,
                'state': state_label, 'ic': ic, 'p': p
            })

res = pd.DataFrame(records)

def best_state_row(group):
    states_only = group[group['state'] != 'Full'].dropna(subset=['ic'])
    if states_only.empty:
        return None
    return states_only.loc[states_only['ic'].abs().idxmax()]

summary = []
for (fac, hz), grp in res.groupby(['factor', 'horizon']):
    full_row  = grp[grp['state'] == 'Full'].iloc[0]
    best_row  = best_state_row(grp)
    if best_row is None:
        continue
    summary.append({
        'factor':     fac,
        'horizon':    hz,
        'ic_full':    full_row['ic'],
        'p_full':     full_row['p'],
        'ic_best':    best_row['ic'],
        'p_best':     best_row['p'],
        'best_state': best_row['state'],
        'abs_gain':   abs(best_row['ic']) - abs(full_row['ic']),
    })

sdf = pd.DataFrame(summary)

hz_order  = ['1D', '5D', '20D']
fac_order = list(factors.keys())
BAR_W = 0.35

state_colors = {
    'S0:HighPrem':   PAPER_BLUE,
    'S1:HardUnwind': PAPER_RED,
    'S2:Stress':     PAPER_ORANGE,
    'S3:Drift':      PAPER_TEAL,
    'S4:Rolldown':   PAPER_PURPLE,
}

def star(p):
    if np.isnan(p): return ''
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '†'
    return ''

panel_letters = {'1D': '(a)', '5D': '(b)', '20D': '(c)'}

fig, axes = plt.subplots(3, 1, figsize=(14, 13))
fig.suptitle(
    'Full-Sample IC vs Best Regime IC — Factor Predictability Lift\n'
    '** p<0.01  * p<0.05  † p<0.10  |  Train Set Only',
    fontsize=18, fontweight='bold', color='#1a1a1a', y=0.99
)

x = np.arange(len(fac_order))

for ax, hz in zip(axes, hz_order):
    sub = sdf[sdf['horizon'] == hz].set_index('factor').reindex(fac_order)

    ic_full = sub['ic_full'].values
    ic_best = sub['ic_best'].values
    p_full  = sub['p_full'].values
    p_best  = sub['p_best'].values
    best_states = sub['best_state'].values

    ax.bar(x - BAR_W/2, ic_full, BAR_W,
           color=PAPER_GRAY, alpha=0.75, label='Full Sample', zorder=3)

    for i, (val, bs) in enumerate(zip(ic_best, best_states)):
        color = state_colors.get(bs, PAPER_GRAY) if isinstance(bs, str) else PAPER_GRAY
        ax.bar(x[i] + BAR_W/2, val, BAR_W, color=color, alpha=0.85, zorder=3)

    # Significance annotations above bars — now bold for contrast
    for i, (vf, pf, vb, pb) in enumerate(zip(ic_full, p_full, ic_best, p_best)):
        sf, sb = star(pf), star(pb)
        offset = 0.01
        if not np.isnan(vf) and sf:
            ax.text(x[i] - BAR_W/2, vf + np.sign(vf)*offset, sf,
                    ha='center', va='bottom' if vf >= 0 else 'top',
                    color='#000000', fontsize=13, fontweight='bold')
        if not np.isnan(vb) and sb:
            ax.text(x[i] + BAR_W/2, vb + np.sign(vb)*offset, sb,
                    ha='center', va='bottom' if vb >= 0 else 'top',
                    color='#000000', fontsize=13, fontweight='bold')

    # Best-state label — anchored at the zero line, offset in points
    # (not data units) so the glyph fully clears axhline(0) regardless
    # of how each panel's y-range happens to be scaled
    for i, (bs, val) in enumerate(zip(best_states, ic_best)):
        if isinstance(bs, str):
            short = bs.split(':')[0]  # "S0", "S1" etc
            is_pos = (not np.isnan(val)) and val >= 0
            ax.annotate(
                short,
                xy=(x[i] + BAR_W / 2, 0),
                xytext=(0, 4 if is_pos else -4),
                textcoords='offset points',
                ha='center',
                va='bottom' if is_pos else 'top',
                fontsize=12, color='#1a1a1a', fontweight='bold', zorder=4
            )

    ax.axhline(0, color='#999999', linewidth=0.8, zorder=2)

    # Lift arrows for gains > 0.10
    for i, gain in enumerate(sub['abs_gain'].values):
        if not np.isnan(gain) and gain > 0.10:
            vf = ic_full[i] if not np.isnan(ic_full[i]) else 0
            vb = ic_best[i] if not np.isnan(ic_best[i]) else 0
            ax.annotate('', xy=(x[i] + BAR_W/2, vb),
                        xytext=(x[i] - BAR_W/2, vf),
                        arrowprops=dict(arrowstyle='->', color='#444444',
                                        lw=1.0, connectionstyle='arc3,rad=0.25'),
                        zorder=5)

    ax.set_xlim(-0.6, len(fac_order) - 0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(fac_order if hz == '20D' else [''] * len(fac_order),
                       fontsize=13)
    ax.set_ylabel(f'IC ({hz})', fontsize=13)
    ax.set_title(f'{panel_letters[hz]} {hz} Horizon', loc='left', fontsize=13)
    ax.tick_params(axis='y', labelsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, color='#e0e0e0', linewidth=0.6, zorder=1)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)

    if hz == '20D':
        patches = [mpatches.Patch(color=PAPER_GRAY, alpha=0.75, label='Full Sample')]
        for sname, scol in state_colors.items():
            patches.append(mpatches.Patch(color=scol, alpha=0.85, label=sname))
        ax.legend(handles=patches, loc='upper right', fontsize=11,
                  frameon=True, facecolor='white', edgecolor='#cccccc')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(RESEARCH_ROOT / 'plots/regime_ic_lift.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")

done


C:\Users\sriva\AppData\Local\Temp\ipykernel_4384\902217663.py:172: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [67]:
import numpy as np
import pandas as pd
from scipy import stats

STATE_IC_5D = {
    0: {'vix_close': +0.44, 'basis': +0.17, 'cost_of_carry': +0.27, 'fii_fut_net_pct': -0.34},
    1: {'basis': -0.32, 'cost_of_carry': -0.28},
    2: {'vix_close': +0.66, 'pcr': -0.34, 'max_pain_dist_pct': -0.40, 'basis': +0.20},
    3: {'pcr': +0.16, 'basis': -0.20},
    4: {'basis': -0.19, 'cost_of_carry': -0.29, 'fut_chng_oi_pct': -0.28,
        'fii_fut_net_pct': -0.45, 'client_fut_net_pct': +0.29},
}

FULL_IC_5D = {
    'vix_close': +0.20, 'basis': -0.25, 'cost_of_carry': -0.17,
    'fut_chng_oi_pct': -0.16, 'client_fut_net_pct': -0.12,
}

STATE_MARKERS = {0: 'o', 1: '^', 2: 's', 3: 'D', 4: 'v'}

# ── Reused verbatim from hmm_regimes.png — same states, same colors ────────
state_colors = {0: PAPER_BLUE, 1: PAPER_RED, 2: PAPER_ORANGE, 3: PAPER_TEAL, 4: PAPER_PURPLE}

# Method-comparison accent — grayscale only, kept entirely separate from
# the state palette above so "Static vs Regime" never visually collides
# with "which state" in panel (c).
METHOD_STATIC = PAPER_GRAY     # '#888888'
METHOD_REGIME = '#3a3a3a'      # dark neutral, not in the state palette

factor_cols = [
    'vix_close', 'pcr', 'max_pain_dist_pct', 'basis',
    'cost_of_carry', 'fut_chng_oi_pct', 'fii_fut_net_pct', 'client_fut_net_pct'
]

train = df[df['split'] == 'train'].copy().reset_index(drop=True)

for col in factor_cols:
    expanding_mean = train[col].expanding().mean()
    expanding_std  = train[col].expanding().std().replace(0, np.nan)
    train[f'z_{col}'] = (train[col] - expanding_mean) / expanding_std

train = train.dropna(subset=[f'z_{col}' for col in factor_cols] + ['fwd_ret_5d', 'hmm_state'])

def regime_score(row):
    weights = STATE_IC_5D.get(int(row['hmm_state']), {})
    if not weights: return np.nan
    score   = sum(row[f'z_{col}'] * w for col, w in weights.items())
    total_w = sum(abs(w) for w in weights.values())
    return score / total_w

def static_score(row):
    score   = sum(row[f'z_{col}'] * w for col, w in FULL_IC_5D.items())
    total_w = sum(abs(w) for w in FULL_IC_5D.values())
    return score / total_w

train['score_regime'] = train.apply(regime_score, axis=1)
train['score_static'] = train.apply(static_score, axis=1)

def eval_composite(scores, returns, label):
    mask = scores.notna() & returns.notna()
    s, r = scores[mask], returns[mask]
    ic, p = stats.spearmanr(s, r)
    quintiles  = pd.qcut(s, 5, labels=False)
    q_means    = r.groupby(quintiles).mean()
    long_mask  = quintiles == 4
    short_mask = quintiles == 0
    ls_ret  = pd.concat([r[long_mask], -r[short_mask]])
    sharpe  = ls_ret.mean() / ls_ret.std() * np.sqrt(252/5)
    hit_rate = (r[long_mask] > 0).mean()
    print(f"\n{label}")
    print(f"  IC={ic:+.3f}  p={p:.4f}")
    print(f"  Q5-Q1 spread: {q_means.iloc[-1]-q_means.iloc[0]:+.2f}%")
    print(f"  L/S Annualized Sharpe: {sharpe:.2f}")
    print(f"  Top-quintile hit rate: {hit_rate:.1%}")
    print(f"  Quintile avg returns: {[f'{v:+.2f}%' for v in q_means]}")
    return {'ic': ic, 'p': p, 'spread': q_means.iloc[-1]-q_means.iloc[0],
            'sharpe': sharpe, 'hit_rate': hit_rate, 'q_means': q_means}

r_regime = eval_composite(train['score_regime'], train['fwd_ret_5d'], 'REGIME-CONDITIONED')
r_static = eval_composite(train['score_static'], train['fwd_ret_5d'], 'STATIC (full-sample)')

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Regime-Conditioned vs Static Composite Score — 5D Forward Return\n'
             'Train Set Only', fontsize=13.5, fontweight='bold', color='#1a1a1a')

q_labels = ['Q1\n(Short)', 'Q2', 'Q3', 'Q4', 'Q5\n(Long)']
x, w = np.arange(5), 0.35

ax = axes[0]
ax.bar(x - w/2, r_static['q_means'], w, color=METHOD_STATIC, alpha=0.85, label='Static')
ax.bar(x + w/2, r_regime['q_means'], w, color=METHOD_REGIME, alpha=0.85, label='Regime')
for i, (sv, rv) in enumerate(zip(r_static['q_means'], r_regime['q_means'])):
    ax.text(i - w/2, sv + np.sign(sv)*0.01, f'{sv:+.2f}', ha='center',
            va='bottom' if sv >= 0 else 'top', color='#1a1a1a', fontsize=7.5)
    ax.text(i + w/2, rv + np.sign(rv)*0.01, f'{rv:+.2f}', ha='center',
            va='bottom' if rv >= 0 else 'top', color='#1a1a1a', fontsize=7.5)
ax.axhline(0, color='#999999', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(q_labels, fontsize=9)
ax.set_ylabel('Avg 5D Return (%)', fontsize=9.5)
ax.set_title('(a) Quintile Returns', loc='left', fontsize=11)
ax.yaxis.grid(True, color='#e0e0e0', lw=0.6); ax.xaxis.grid(False)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=9)

ax = axes[1]
metrics = ['IC', 'Sharpe', 'Hit Rate']
s_vals  = [r_static['ic'], r_static['sharpe'], r_static['hit_rate']]
r_vals  = [r_regime['ic'],  r_regime['sharpe'],  r_regime['hit_rate']]
x2 = np.arange(3)
ax.bar(x2 - w/2, s_vals, w, color=METHOD_STATIC, alpha=0.85, label='Static')
ax.bar(x2 + w/2, r_vals, w, color=METHOD_REGIME, alpha=0.85, label='Regime')
for i, (sv, rv) in enumerate(zip(s_vals, r_vals)):
    ax.text(i - w/2, sv + 0.05, f'{sv:.2f}', ha='center', color='#1a1a1a', fontsize=7.5)
    ax.text(i + w/2, rv + 0.05, f'{rv:.2f}', ha='center', color='#1a1a1a', fontsize=7.5)
ax.axhline(0, color='#999999', lw=0.8)
ax.set_xticks(x2); ax.set_xticklabels(metrics, fontsize=9.5)
ax.set_title('(b) Summary Metrics', loc='left', fontsize=11)
ax.yaxis.grid(True, color='#e0e0e0', lw=0.6); ax.xaxis.grid(False)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=9)

ax = axes[2]
for state_id, color in state_colors.items():
    mask = train['hmm_state'] == state_id
    ax.scatter(train.loc[mask, 'score_regime'], train.loc[mask, 'fwd_ret_5d'],
               color=color, marker=STATE_MARKERS[state_id],
               alpha=0.8, s=28, edgecolors='none',
               label=f'S{state_id}')
ax.axhline(0, color='#999999', lw=0.8)
ax.axvline(0, color='#999999', lw=0.8, linestyle='--')
ax.set_xlabel('Regime Composite Score', fontsize=9.5)
ax.set_ylabel('5D Forward Return (%)', fontsize=9.5)
ax.set_title('(c) Score vs Return by State', loc='left', fontsize=11)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=8.5)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(RESEARCH_ROOT / 'plots/regime_composite_vs_static.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")


REGIME-CONDITIONED
  IC=+0.262  p=0.0000
  Q5-Q1 spread: +1.46%
  L/S Annualized Sharpe: 2.54
  Top-quintile hit rate: 68.1%
  Quintile avg returns: ['-0.54%', '-0.19%', '+0.16%', '+0.34%', '+0.92%']

STATIC (full-sample)
  IC=+0.314  p=0.0000
  Q5-Q1 spread: +1.86%
  L/S Annualized Sharpe: 3.03
  Top-quintile hit rate: 74.5%
  Quintile avg returns: ['-0.55%', '-0.57%', '+0.12%', '+0.37%', '+1.31%']
done


C:\Users\sriva\AppData\Local\Temp\ipykernel_4384\619878711.py:143: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [74]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

n_boot = 2000
rng = np.random.default_rng(42)

def bootstrap_metrics(scores, returns, n_boot=2000):
    n = len(scores)
    ics, sharpes, hit_rates, spreads = [], [], [], []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        s, r = scores.iloc[idx], returns.iloc[idx]

        ic, _ = spearmanr(s, r)

        quintiles = pd.qcut(s, 5, labels=False, duplicates='drop')
        q_means = r.groupby(quintiles).mean()
        if len(q_means) < 5:
            continue
        spread = q_means.iloc[-1] - q_means.iloc[0]

        long_mask  = quintiles == 4
        short_mask = quintiles == 0
        ls_ret = pd.concat([r[long_mask], -r[short_mask]])
        sharpe = ls_ret.mean() / ls_ret.std() * np.sqrt(252/5)
        hit    = (r[long_mask] > 0).mean()

        ics.append(ic)
        sharpes.append(sharpe)
        hit_rates.append(hit)
        spreads.append(spread)

    return {
        'ic':       np.array(ics),
        'sharpe':   np.array(sharpes),
        'hit_rate': np.array(hit_rates),
        'spread':   np.array(spreads),
    }

mask = train['score_regime'].notna() & train['score_static'].notna() & train['fwd_ret_5d'].notna()
t = train[mask].reset_index(drop=True)

print("Bootstrapping regime composite...")
boot_r = bootstrap_metrics(t['score_regime'], t['fwd_ret_5d'], n_boot)
print("Bootstrapping static composite...")
boot_s = bootstrap_metrics(t['score_static'], t['fwd_ret_5d'], n_boot)

diff_ic      = boot_r['ic']       - boot_s['ic']
diff_sharpe  = boot_r['sharpe']   - boot_s['sharpe']
diff_hit     = boot_r['hit_rate'] - boot_s['hit_rate']
diff_spread  = boot_r['spread']   - boot_s['spread']

def ci95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

def p_gt_zero(arr):
    return (arr > 0).mean()

print("\n=== BOOTSTRAP RESULTS (n=2000, 95% CI) ===")
for name, b_r, b_s, diff in [
    ('IC',        boot_r['ic'],       boot_s['ic'],       diff_ic),
    ('Sharpe',    boot_r['sharpe'],   boot_s['sharpe'],   diff_sharpe),
    ('Hit Rate',  boot_r['hit_rate'], boot_s['hit_rate'], diff_hit),
    ('Q5-Q1 Spd', boot_r['spread'],  boot_s['spread'],   diff_spread),
]:
    lo_r, hi_r = ci95(b_r)
    lo_s, hi_s = ci95(b_s)
    lo_d, hi_d = ci95(diff)
    p = p_gt_zero(diff)
    print(f"\n{name}:")
    print(f"  Regime: {np.mean(b_r):.3f}  95% CI [{lo_r:.3f}, {hi_r:.3f}]")
    print(f"  Static: {np.mean(b_s):.3f}  95% CI [{lo_s:.3f}, {hi_s:.3f}]")
    print(f"  Diff:   {np.mean(diff):+.3f}  95% CI [{lo_d:.3f}, {hi_d:.3f}]  P(regime>static)={p:.3f}")

# --- Plot ---
# Reused from regime_composite_vs_static.png — same two methods being compared
METHOD_STATIC = PAPER_GRAY     # '#888888'
METHOD_REGIME = '#3a3a3a'      # dark neutral, distinct from the per-state palette

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Bootstrap CI — Regime vs Static Composite (n=2000)\n'
             '5D Forward Return | Train Set Only',
             fontsize=13.5, fontweight='bold', color='#1a1a1a')

panel_letters = ['(a)', '(b)', '(c)', '(d)']
pairs = [
    ('IC',         diff_ic,     boot_r['ic'],     boot_s['ic']),
    ('Sharpe',     diff_sharpe, boot_r['sharpe'], boot_s['sharpe']),
    ('Hit Rate',   diff_hit,    boot_r['hit_rate'], boot_s['hit_rate']),
    ('Q5-Q1 Spd',  diff_spread, boot_r['spread'], boot_s['spread']),
]

for letter, ax, (name, diff, br, bs) in zip(panel_letters, axes.flat, pairs):
    lo, hi = ci95(diff)
    p = p_gt_zero(diff)

    ax.hist(bs, bins=60, color=METHOD_STATIC, alpha=0.55, density=True, label='Static')
    ax.hist(br, bins=60, color=METHOD_REGIME, alpha=0.55, density=True, label='Regime')
    ax.set_ylabel('Density', fontsize=9)

    ax2 = ax.twinx()
    ax2.hist(diff, bins=60, color=PAPER_TEAL, alpha=0.35, density=True, label='Diff')
    ax2.axvline(0, color='#999999', lw=0.8, ls='--')
    ax2.axvline(lo, color=PAPER_TEAL, lw=0.8, ls=':')
    ax2.axvline(hi, color=PAPER_TEAL, lw=0.8, ls=':')
    ax2.axvline(np.mean(diff), color=PAPER_TEAL, lw=1.6)
    ax2.set_yticks([])
    ax2.grid(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    ci_str = f'[{lo:+.3f}, {hi:+.3f}]'
    significant = lo > 0 or hi < 0
    if lo > 0:
        sig_str, sig_color = '✓ significant (regime > static)', PAPER_TEAL
    elif hi < 0:
        sig_str, sig_color = '✓ significant (static > regime)', PAPER_RED
    else:
        sig_str, sig_color = '~ overlaps zero', '#777777'

    title_weight = 'bold' if significant else 'normal'
    title_color  = sig_color if significant else '#1a1a1a'

    ax.set_title(f'{letter} {name}  —  Diff 95% CI: {ci_str}\n{sig_str}',
                 loc='left', fontsize=10.5, color=title_color,
                 fontweight=title_weight, pad=10)

    ax.set_xlabel('Value', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=METHOD_STATIC, alpha=0.55),
        plt.Rectangle((0, 0), 1, 1, color=METHOD_REGIME, alpha=0.55),
        plt.Rectangle((0, 0), 1, 1, color=PAPER_TEAL, alpha=0.35),
    ]
    ax.legend(handles, ['Static', 'Regime', 'Diff'], loc='upper left',
              frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=8.5)

    ax.text(0.97, 0.05, f'P(R>S)={p:.3f}', transform=ax.transAxes,
            ha='right', fontsize=8.5, color=title_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor='#cccccc', linewidth=0.6))

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(RESEARCH_ROOT / 'plots/bootstrap_regime_vs_static.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("done")

Bootstrapping regime composite...
Bootstrapping static composite...

=== BOOTSTRAP RESULTS (n=2000, 95% CI) ===

IC:
  Regime: 0.262  95% CI [0.141, 0.384]
  Static: 0.314  95% CI [0.187, 0.431]
  Diff:   -0.051  95% CI [-0.223, 0.117]  P(regime>static)=0.277

Sharpe:
  Regime: 2.596  95% CI [1.149, 4.073]
  Static: 3.296  95% CI [1.873, 4.729]
  Diff:   -0.700  95% CI [-2.745, 1.373]  P(regime>static)=0.252

Hit Rate:
  Regime: 0.679  95% CI [0.532, 0.822]
  Static: 0.755  95% CI [0.617, 0.872]
  Diff:   -0.076  95% CI [-0.273, 0.106]  P(regime>static)=0.201

Q5-Q1 Spd:
  Regime: 1.493  95% CI [0.639, 2.366]
  Static: 2.003  95% CI [1.120, 2.961]
  Diff:   -0.510  95% CI [-1.788, 0.724]  P(regime>static)=0.208
done


C:\Users\sriva\AppData\Local\Temp\ipykernel_4384\3175652254.py:150: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [52]:
con.close()